# GRUEncoder Training — SOLUSDT Phase A (h32 variant)

Trains the GRUEncoder (hidden=32, dropout=0.2, 5K params) on processed
feature data using a free T4 GPU.

## Iteration workflow
1. **Code changes** → `git push` to GitHub
2. **Colab**: run cells below (clone/pull code, train, save checkpoint to Drive)
3. **Local**: run `scripts/pull_checkpoint.py` to sync checkpoint from Drive → UI

Data stays on Drive (upload once). Only code changes get pushed to git.

In [ ]:
# Cell 1: Mount Google Drive (stores data + checkpoints)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: Install Python dependencies
!pip install --quiet pandas pyarrow pyyaml httpx numba
print('Dependencies installed')

In [ ]:
# Cell 4: Clone / pull project code from GitHub
import os

REPO_URL = 'https://github.com/sandeep999-cyber/solusdt-ml-trader.git'
PROJECT_DIR = '/content/ModelProject'

if os.path.exists(PROJECT_DIR):
    # Already cloned — pull latest
    %cd "$PROJECT_DIR"
    !git pull
    print('Pulled latest code from GitHub')
else:
    !git clone "$REPO_URL" "$PROJECT_DIR"
    %cd "$PROJECT_DIR"
    print('Cloned repository from GitHub')

# Symlink data from Drive into the project tree
DRIVE_DATA = '/content/drive/MyDrive/ModelProject/data_processed'
LOCAL_DATA = 'data/processed/v1/SOLUSDT/1m'

if not os.path.exists(LOCAL_DATA):
    if not os.path.exists(DRIVE_DATA):
        raise FileNotFoundError(
            f'Processed data not found on Drive at {DRIVE_DATA}. '
            'Upload data/processed/ to MyDrive/ModelProject/data_processed/ first.'
        )
    os.makedirs(os.path.dirname(LOCAL_DATA), exist_ok=True)
    # Use symlink for speed; fall back to copy if symlink fails
    try:
        os.symlink(DRIVE_DATA, LOCAL_DATA)
        print(f'Symlinked data from Drive: {DRIVE_DATA} -> {LOCAL_DATA}')
    except AttributeError:
        # Windows may not support symlinks in Colab
        import shutil
        shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
        print(f'Copied data from Drive to {LOCAL_DATA}')
else:
    print(f'Data already available at {LOCAL_DATA}')

print(f'\nProject ready at {PROJECT_DIR}')
print(f'Contents: {os.listdir(".")}')

In [ ]:
# Cell 5: Smoke test only (~10s)
# Run a 2-epoch smoke test to verify everything works before the long run.
# This does NOT run full training — that's Cell 6.
smoke_ok = False
try:
    !python -m model.train --config configs/phase_a_gru_h32.yaml --smoke-test-only
    smoke_ok = True
    print('\nSmoke test PASSED — ready for full training.')
except Exception as e:
    print(f'Smoke test FAILED: {e}')

In [ ]:
# Cell 6: Full training run
# Takes ~3-5 minutes on a T4 GPU (vs ~2h on CPU)
if smoke_ok:
    !python -m model.train --config configs/phase_a_gru_h32.yaml
else:
    print('Skipping full training — smoke test did not pass. Fix errors first.')

In [ ]:
# Cell 7: Save checkpoint to Drive at a fixed path
import json
from pathlib import Path

RUNS_DIR = Path('model/runs')
run_dirs = sorted(RUNS_DIR.iterdir())
if not run_dirs:
    raise RuntimeError('No training runs found')

# Get the most recent non-smoketest run
real_runs = [d for d in run_dirs if d.is_dir() and 'smoketest' not in d.name]
if not real_runs:
    real_runs = [d for d in run_dirs if d.is_dir()]
latest_run = real_runs[-1].name
print(f'Latest run: {latest_run}')

# --- Save checkpoint + metrics to a stable Drive path ---
DRIVE_RUN = Path('/content/drive/MyDrive/ModelProject/checkpoints')
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

best_path = RUNS_DIR / latest_run / 'checkpoints' / 'best.pt'
if best_path.exists():
    import shutil
    # Copy to stable path (always overwrites "latest")
    shutil.copy(best_path, DRIVE_RUN / 'best.pt')
    print(f'Saved best.pt -> {DRIVE_RUN / "best.pt"}')
    # Also archive with run name
    shutil.copy(best_path, DRIVE_RUN / f'{latest_run}_best.pt')
    print(f'Archived -> {DRIVE_RUN / f"{latest_run}_best.pt"}')
else:
    print('No best.pt found')

# Save pointer file (run_name + timestamp) for local sync script
from datetime import datetime, timezone
pointer = {
    'run_name': latest_run,
    'checkpoint_path': str(DRIVE_RUN / f'{latest_run}_best.pt'),
    'updated': datetime.now(timezone.utc).isoformat(),
}
with open(DRIVE_RUN / 'latest.json', 'w') as f:
    json.dump(pointer, f, indent=2)
print(f'Saved pointer -> {DRIVE_RUN / "latest.json"}')

# Save metrics
metrics_path = RUNS_DIR / latest_run / 'metrics.jsonl'
if metrics_path.exists():
    shutil.copy(metrics_path, DRIVE_RUN / f'{latest_run}_metrics.jsonl')
    print(f'Saved metrics -> {DRIVE_RUN / f"{latest_run}_metrics.jsonl"}')
    # Print final metrics
    with open(metrics_path) as f:
        lines = f.readlines()
    last = json.loads(lines[-1])
    print(f'\nFinal epoch: {last.get("epoch", "?")}')
    print(f'Val loss:   {last.get("loss", "?")}')
    print(f'NLL:        {last.get("nll", "?")}')
    print(f'Baseline:   {last.get("baseline_loss", "?")}')
    print(f'DeltaBL:    {last.get("baseline_delta", "?")}')

In [ ]:
# Cell 8: Download checkpoint to local machine
from google.colab import files

best_path = RUNS_DIR / latest_run / 'checkpoints' / 'best.pt'
if best_path.exists():
    files.download(str(best_path))
    print('Downloading best.pt...')
else:
    print('No best.pt to download')